# Лабораторная работа 14. Reward model и DPO (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 5:** дообучение и выравнивание

**Цель.** Реализовать с нуля (без `trl`) обучение модели вознаграждения по модели Брэдли–Терри и Direct Preference Optimization, изучить динамику DPO, best-of-n и смещение по длине.

**Что нужно сдать:**
1. Reward model с точностью на отложенных парах.
2. DPO с логированием неявного вознаграждения, margin, log-likelihood выбранных и отвергнутых ответов и оценки KL.
3. Сравнение \(\beta \in \{0.01, 0.05, 0.1, 0.5, 1.0\}\) (в `FAST_DEV_RUN` — 0.1 и 0.5).
4. Best-of-n с reward model в сравнении с оракулом.
5. Эксперимент со смещением reward model по длине.

**Данные.** Синтетические задачи из ЛР 13 с проверяемым ответом. Выбранный ответ — правильный, отвергнутый — правдоподобная ошибка (сумма с ошибкой, сортировка с перестановкой и т. п.). Так можно точно измерять качество политики.

**Теория.**
- **Брэдли–Терри:** \(P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))\), loss \(-\log\sigma(r_w - r_l)\).
- **RLHF-цель:** \(\max_\pi \mathbb{E}[r(x,y)] - \beta\,\mathrm{KL}(\pi\|\pi_{ref})\), её оптимум \(\pi^*(y|x) \propto \pi_{ref}(y|x)\exp(r(x,y)/\beta)\).
- Отсюда \(r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta\log Z(x)\), и подстановка в Брэдли–Терри даёт **DPO** (Rafailov et al., 2023):
\[
\mathcal{L}_{DPO} = -\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\Big).
\]
Величина \(\hat r(x,y) = \beta\log\frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}\) — неявное вознаграждение.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
# ==== Загрузка предобученных моделей Hugging Face ====
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_lm(name, dtype=None):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    dtype = dtype or (torch.float16 if DEVICE == "cuda" else torch.float32)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    return model.to(DEVICE).eval(), tok

@torch.no_grad()
def perplexity(model, tok, text, max_len=512, stride=256, max_windows=None):
    ids = tok(text, return_tensors="pt").input_ids.to(DEVICE)
    nll, cnt, prev = 0.0, 0, 0
    for i, start in enumerate(range(0, ids.size(1) - 1, stride)):
        if max_windows and i >= max_windows: break
        end = min(start + max_len, ids.size(1)); x = ids[:, start:end]
        tgt = x.clone(); tgt[:, : max(0, prev - start)] = -100
        out = model(x, labels=tgt)
        n = (tgt[:, 1:] != -100).sum().item(); nll += out.loss.float().item() * n; cnt += n; prev = end
        if end == ids.size(1): break
    return math.exp(nll / cnt)

In [ ]:
# ==== Из ЛР 13: синтетические инструкции, LoRA, SFT (эталонное решение) ====
import copy
WORDS = ("apple river garden window silver planet orange monkey castle pencil bridge forest winter summer rocket "
         "dragon basket mirror candle thunder library coffee guitar island pirate jungle ladder magnet needle ocean "
         "puzzle rabbit saddle tomato violin wizard yellow zebra anchor butter circle").split()
TEMPLATE = "### Instruction:\n{inst}\n### Response:\n"

def make_example(rng):
    kind = rng.choice(["reverse", "sort", "upper", "add"])
    if kind == "reverse":
        w = rng.choice(WORDS); return kind, f"Reverse the letters of the word: {w}", w[::-1]
    if kind == "sort":
        xs = rng.sample(range(1, 100), rng.randint(3, 5)); return kind, "Sort in ascending order: " + ", ".join(map(str, xs)), ", ".join(map(str, sorted(xs)))
    if kind == "upper":
        ph = " ".join(rng.sample(WORDS, 2)); return kind, f"Convert to uppercase: {ph}", ph.upper()
    a, b = rng.randint(10, 99), rng.randint(10, 99); return kind, f"What is {a} plus {b}?", str(a + b)

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        for p in self.base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features, device=base.weight.device) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device))
    def forward(self, x): return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale
    def merge(self):
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(self.base.weight.device)
        with torch.no_grad():
            lin.weight.copy_(self.base.weight + self.scale * self.B @ self.A)
            if self.base.bias is not None: lin.bias.copy_(self.base.bias)
        return lin

ATTN = ("q_proj", "k_proj", "v_proj", "o_proj"); MLP = ("gate_proj", "up_proj", "down_proj")

def inject_lora(model, targets=ATTN + MLP, r=8, alpha=16):
    for p in model.parameters(): p.requires_grad_(False)
    for _, mod in list(model.named_modules()):
        for cn, ch in list(mod.named_children()):
            if isinstance(ch, nn.Linear) and cn.endswith(targets): setattr(mod, cn, LoRALinear(ch, r, alpha))
    return model

def merge_lora(model):
    for _, mod in list(model.named_modules()):
        for cn, ch in list(mod.named_children()):
            if isinstance(ch, LoRALinear): setattr(mod, cn, ch.merge())
    return model

def encode_pairs(tok, prompts, responses, mask_prompt=True):
    """Склейка промпт+ответ+eos, метки промпта = -100, паддинг справа."""
    seqs, labels = [], []
    for p_text, r_text in zip(prompts, responses):
        p = tok(p_text, add_special_tokens=False).input_ids
        r = tok(r_text, add_special_tokens=False).input_ids + [tok.eos_token_id]
        seqs.append(p + r); labels.append(([-100] * len(p) if mask_prompt else p) + r)
    T = max(map(len, seqs))
    x = torch.full((len(seqs), T), tok.pad_token_id); y = torch.full((len(seqs), T), -100); att = torch.zeros(len(seqs), T, dtype=torch.long)
    for i, (s, l) in enumerate(zip(seqs, labels)):
        x[i, :len(s)] = torch.tensor(s); y[i, :len(l)] = torch.tensor(l); att[i, :len(s)] = 1
    return x.to(DEVICE), y.to(DEVICE), att.to(DEVICE)

def sft(model, tok, data, steps, lr=1e-3, B=8, seed=0):
    set_seed(seed); params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0); rng = random.Random(seed); model.train()
    for _ in range(steps):
        batch = rng.sample(data, B)
        x, y, att = encode_pairs(tok, [TEMPLATE.format(inst=e[1]) for e in batch], [e[2] for e in batch])
        loss = model(input_ids=x, attention_mask=att, labels=y).loss
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
    model.eval(); return model

@torch.no_grad()
def answer(model, tok, inst, max_new=12, **gen_kw):
    ids = tok(TEMPLATE.format(inst=inst), return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)
    out = model.generate(ids, max_new_tokens=max_new, pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id, **gen_kw)
    return [tok.decode(o[ids.size(1):], skip_special_tokens=True).split("\n")[0].strip() for o in out]

def evaluate(model, tok, data, **gen_kw):
    res = {}
    for kind, inst, resp in data:
        res.setdefault(kind, []).append(answer(model, tok, inst, do_sample=False, **gen_kw)[0] == resp)
    out = {k: float(np.mean(v)) for k, v in res.items()}; out["всего"] = float(np.mean([x for v in res.values() for x in v]))
    return out

In [ ]:
BASE = "HuggingFaceTB/SmolLM2-135M" if FAST_DEV_RUN else "Qwen/Qwen2.5-0.5B"
base, tok = load_lm(BASE, dtype=torch.float32)
rng = random.Random(0); train_tasks = [make_example(rng) for _ in range(2000)]
rng = random.Random(1); test_tasks = [make_example(rng) for _ in range(40 if FAST_DEV_RUN else 300)]

# Референсная политика: короткий SFT, чтобы модель знала формат, но ошибалась (иначе DPO нечего улучшать)
t0 = time.time()
ref = merge_lora(sft(inject_lora(copy.deepcopy(base), r=8), tok, train_tasks, steps=40 if FAST_DEV_RUN else 150))
for p in ref.parameters(): p.requires_grad_(False)
ref_eval = evaluate(ref, tok, test_tasks)
print(f"π_ref за {time.time() - t0:.0f} c:", {k: round(v, 2) for k, v in ref_eval.items()})

## 0. Пары предпочтений

Реализуйте `corrupt(kind, inst, resp, rng)`, возвращающую правдоподобный неверный ответ того же формата:
- `add` — сумма со сдвигом на ±1…10;
- `sort` — перестановка двух соседних чисел;
- `upper` — одно из слов в нижнем регистре;
- `reverse` — исходное слово или слово с двумя переставленными буквами.

In [ ]:
def corrupt(kind, inst, resp, rng):
    # TODO: ваш код
    raise NotImplementedError

def make_pairs(tasks, seed, verbose_bias=None):
    """verbose_bias=p: с вероятностью p к выбранному ответу добавляется «вежливый хвост», с той же вероятностью отвергнутый остаётся кратким."""
    rng, pairs = random.Random(seed), []
    for kind, inst, resp in tasks:
        ch, rj = resp, corrupt(kind, inst, resp, rng)
        if verbose_bias is not None:
            if rng.random() < verbose_bias: ch = ch + TAIL
            else: rj = rj + TAIL
        pairs.append((TEMPLATE.format(inst=inst), ch, rj))
    return pairs

TAIL = " I hope this helps! Let me know if you need anything else."
N_PAIRS = 240 if FAST_DEV_RUN else 3000
train_pairs = make_pairs(train_tasks[:N_PAIRS], 0)
test_pairs = make_pairs(test_tasks, 1)
print(train_pairs[0]); print(train_pairs[1])

## 1. Reward model (Брэдли–Терри)

Reward model: та же предобученная сеть, скалярная голова поверх скрытого состояния **последнего непаддингового токена**, LoRA на всех линейных слоях. Обучите на `train_pairs` и посчитайте точность \(\mathbb{1}[r_w > r_l]\) на `test_pairs`.

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, lm):
        super().__init__()
        self.lm = inject_lora(lm, r=8)
        self.head = nn.Linear(lm.config.hidden_size, 1).to(DEVICE)
    def forward(self, x, att):
        # TODO: ваш код
        raise NotImplementedError

def rm_scores(rm, prompts, responses, bs=16):
    out = []
    with torch.no_grad():
        for i in range(0, len(prompts), bs):
            x, _, att = encode_pairs(tok, prompts[i:i + bs], responses[i:i + bs]); out.append(rm(x, att))
    return torch.cat(out)

def rm_accuracy(rm, pairs):
    rw = rm_scores(rm, [p for p, _, _ in pairs], [c for _, c, _ in pairs]); rl = rm_scores(rm, [p for p, _, _ in pairs], [r for _, _, r in pairs])
    return (rw > rl).float().mean().item()

def train_rm(pairs, steps, B=8, lr=5e-4, seed=0):
    set_seed(seed); rm = RewardModel(copy.deepcopy(base)); rm.train()
    params = [p for p in rm.parameters() if p.requires_grad]; opt = torch.optim.AdamW(params, lr=lr); rng = random.Random(seed); hist = []
    for s in range(steps):
        batch = rng.sample(pairs, B)
        # TODO: ваш код
        raise NotImplementedError
        hist.append(loss.item())
    rm.eval(); return rm, hist

RM_STEPS = 60 if FAST_DEV_RUN else 800
t0 = time.time(); rm, rm_hist = train_rm(train_pairs, RM_STEPS)
print(f"RM: {time.time() - t0:.0f} c, точность на тесте = {rm_accuracy(rm, test_pairs):.3f}")
plt.figure(figsize=(6, 3)); plt.plot(np.convolve(rm_hist, np.ones(10) / 10, "valid")); plt.title("loss reward model"); plt.show()

## 2. DPO с нуля

1. Посчитайте \(\log\pi(y|x)\) — сумму log-вероятностей токенов ответа (промпт и паддинг замаскированы).
2. Предвычислите \(\log\pi_{ref}\) для всех обучающих пар один раз.
3. Политика — копия \(\pi_{ref}\) с LoRA. Логируйте на каждом шаге: loss, accuracy (неявное вознаграждение выбранного больше), margin \(\hat r_w - \hat r_l\), средние \(\log\pi(y_w)\) и \(\log\pi(y_l)\).
4. После обучения оцените exact match и \(\mathrm{KL}(\pi\|\pi_{ref})\) методом Монте-Карло по сэмплам политики: \(\frac1N\sum_i[\log\pi(y_i|x_i) - \log\pi_{ref}(y_i|x_i)]\).

In [ ]:
def seq_logp(model, x, y, att):
    # TODO: ваш код
    raise NotImplementedError

def pair_logps(model, pairs, grad=False):
    prompts = [p for p, _, _ in pairs] * 2; resps = [c for _, c, _ in pairs] + [r for _, _, r in pairs]
    x, y, att = encode_pairs(tok, prompts, resps)
    with torch.set_grad_enabled(grad): lp = seq_logp(model, x, y, att)
    return lp[:len(pairs)], lp[len(pairs):]

ref_lp = []
for i in range(0, len(train_pairs), 16): ref_lp.append(torch.stack(pair_logps(ref, train_pairs[i:i + 16]), 1))
ref_lp = torch.cat(ref_lp)                                 # (N, 2): log π_ref(y_w), log π_ref(y_l)

def dpo(beta, steps, B=8, lr=5e-4, seed=0):
    set_seed(seed); pol = inject_lora(copy.deepcopy(ref), r=8); pol.train()
    params = [p for p in pol.parameters() if p.requires_grad]; opt = torch.optim.AdamW(params, lr=lr); rng = np.random.default_rng(seed); log = []
    for s in range(steps):
        idx = rng.choice(len(train_pairs), B, replace=False)
        # TODO: ваш код
        raise NotImplementedError
    pol.eval(); return pol, pd.DataFrame(log)

@torch.no_grad()
def mc_kl(pol, tasks, n=1):
    vals = []
    for _, inst, _ in tasks:
        prompt = TEMPLATE.format(inst=inst)
        for s in answer(pol, tok, inst, do_sample=True, temperature=1.0, num_return_sequences=n):
            x, y, att = encode_pairs(tok, [prompt], [s]); vals.append((seq_logp(pol, x, y, att) - seq_logp(ref, x, y, att)).item())
    return float(np.mean(vals))

import pandas as pd
DPO_STEPS = 40 if FAST_DEV_RUN else 500
BETAS = [0.1, 0.5] if FAST_DEV_RUN else [0.01, 0.05, 0.1, 0.5, 1.0]
logs, rows, policies = {}, [{"β": "π_ref", **{k: round(v, 3) for k, v in ref_eval.items()}, "KL": 0.0}], {}
for beta in BETAS:
    t0 = time.time(); pol, lg = dpo(beta, DPO_STEPS); logs[beta] = lg; policies[beta] = pol
    rows.append({"β": beta, **{k: round(v, 3) for k, v in evaluate(pol, tok, test_tasks).items()}, "KL": round(mc_kl(pol, test_tasks[:10 if FAST_DEV_RUN else 50]), 2)})
    print(f"β={beta}: {time.time() - t0:.0f} c")
dpo_table = pd.DataFrame(rows); display(dpo_table)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.5))
for beta, lg in logs.items():
    sm = lg.rolling(10, min_periods=1).mean()
    axes[0].plot(sm["loss"], label=f"β={beta}"); axes[1].plot(sm["margin"]); axes[2].plot(sm["logp_w"]); axes[3].plot(sm["logp_l"])
for a, t in zip(axes, ["DPO loss", "margin неявного вознаграждения", "log π(y_w)", "log π(y_l)"]): a.set_title(t); a.set_xlabel("шаг")
axes[0].legend(); plt.tight_layout(); plt.show()

## 3. Best-of-n

Для каждой тестовой задачи сгенерируйте \(n = 8\) (в `FAST_DEV_RUN` — 4) ответов из \(\pi_{ref}\) (temperature 1) и выберите ответ с максимальным вознаграждением reward model. Постройте зависимость exact match от \(n \in \{1, 2, 4, 8\}\) для выбора по RM и для оракула (pass@n: хотя бы один из \(n\) верен). Сравните с лучшей DPO-политикой. Теоретическая оценка KL для best-of-n: \(\log n - (n-1)/n\).

In [ ]:
N_BON = 4 if FAST_DEV_RUN else 8
# TODO: ваш код (сэмплы, оценки RM, кривые best-of-n и pass@n)

## 4. Смещение reward model по длине

Обучите вторую reward model на данных, где в 90% пар к выбранному ответу добавлен «вежливый хвост» `TAIL`, а в 10% — к отвергнутому. Тогда длина почти идеально предсказывает предпочтение. Сравните две RM на тестовых парах трёх типов:
1. без хвостов;
2. хвост у правильного ответа;
3. хвост у неправильного ответа (решающий тест: «честная» RM должна по-прежнему выбирать правильный).

In [ ]:
biased_pairs = make_pairs(train_tasks[:N_PAIRS], 0, verbose_bias=0.9)
t0 = time.time(); rm_biased, _ = train_rm(biased_pairs, RM_STEPS); print(f"RM со смещением: {time.time() - t0:.0f} c")
# TODO: ваш код (три типа тестовых пар и таблица точностей двух RM)

## Вопросы

1. Выведите оптимальную политику для RLHF-цели с KL-регуляризацией и получите из неё DPO loss. Почему \(Z(x)\) сокращается?
2. Опишите типичную динамику DPO: почему \(\log\pi(y_w)\) часто **убывает**, хотя loss падает? Чем это опасно?
3. Как \(\beta\) влияет на скорость роста margin, KL и качество? Что происходит при очень малом \(\beta\)?
4. Почему best-of-n — сильный базовый метод и как растёт его «стоимость» в KL с \(n\)? Чем ограничено качество best-of-n при неидеальной RM (сравните с оракулом)?
5. Как смещение по длине возникает в реальных данных предпочтений и какие методы с ним борются (нормировка на длину, SimPO, штрафы, контроль длины при оценке)?